# Architektura Aplikacji w Pythonie - Zestaw Zaliczeniowy

**WSEI Kraków · semestr letni 2026 · prowadzący: Michał Madejski**

---

## Filozofia tego zestawu

Sześć laboratoriów dało Ci sześć narzędzi. Ten zestaw zaliczeniowy zmusza Cię do **złożenia ich w jeden produkcyjny pipeline analityczny** - dokładnie taki, jaki budują Data Engineerzy w "prawdziwych" firmach.

**Wspólny dataset:** [`stanfordnlp/imdb`](https://huggingface.co/datasets/stanfordnlp/imdb) z Hugging Face Hub - 50 000 recenzji filmów z etykietami sentymentu (pozytywna / negatywna).

**Reguły:**
1. Każdy lab ma blok: **Teoria → Przykład rozwiązany → Zadanie samodzielne**.
2. Zadania samodzielne **rozszerzają** przykład - dokładnie ten sam pattern, inny scenariusz.
3. Cały notebook ma być **uruchamialny od góry do dołu**. Brak hardkodowanych ścieżek, brak ręcznych downloadów.
4. Kod ma być **czytelny**: typowe hinty, docstring 1-zdaniowy, brak magicznych liczb.

**Ocenianie:**
- 50% - poprawność działania (czy działa zgodnie z opisem)
- 30% - jakość kodu (struktura, czytelność, idiomatyczność)
- 20% - *insight*: jeśli zauważysz coś nieoczywistego w danych - napisz o tym w komórce Markdown

---

## Mapa zestawu

| # | Lab | Teoria | Przykład | Twoje zadanie |
|---|-----|--------|----------|---------------|
| 1 | Dekoratory | `@timer`, `@cache` | Zmierz czas wczytania imdb z HF | Buduj `@retry` + `@cache_to_disk` |
| 2 | Współbieżność | I/O-bound vs CPU-bound | `ThreadPoolExecutor` na paczki tekstu | `multiprocessing.Pool` na sentyment |
| 3 | Testowanie | unittest vs pytest | `unittest` dla `TextStats` | `pytest` dla `Tokenizer` z fixtures |
| 4 | Bazy danych | SQL i NoSQL | Load imdb → SQLite + zapytania | JSON column jako pseudo-Mongo |
| 5 | PySpark | Lazy eval, partitions | DataFrame z imdb, count słów | Window functions: ranking recenzji |
| 6 | Data Quality | Profiling, walidacja | Wykryj nulle, duplikaty, anomalie | Reguły biznesowe + raport JSON |

---

## Setup

In [1]:
# Globalna konfiguracja -- jedna komorka, jeden raz
import os, sys, time, json, warnings, random
from pathlib import Path
warnings.filterwarnings("ignore")

WORKDIR = Path("./_workspace")
WORKDIR.mkdir(exist_ok=True)

# Tame log spamu HF Datasets
os.environ.setdefault("HF_DATASETS_DISABLE_PROGRESS_BAR", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

print(f"Python: {sys.version.split()[0]}")
print(f"Workspace: {WORKDIR.resolve()}")

Python: 3.13.12
Workspace: C:\Users\home\Desktop\Python-sk\AAP_LAB_TASKS\zestaw_zaliczeniowy\_workspace


---

# Lab 1 - Dekoratory

## Teoria w trzech zdaniach

**Dekorator** to funkcja, która przyjmuje funkcję i zwraca funkcję. Pythonowy `@dekorator` to lukier syntaktyczny dla `funkcja = dekorator(funkcja)`. Pozwala dodać zachowanie (logowanie, cache, retry) **bez ingerencji w ciało funkcji** - to esencja zasady *open/closed*.

### Wzorzec dekoratora z argumentami

```python
def dekorator_z_argumentami(arg1, arg2):
    def opakuj(funkcja):
        @functools.wraps(funkcja)
        def wrapper(*args, **kwargs):
            # przed wywolaniem
            wynik = funkcja(*args, **kwargs)
            # po wywolaniu
            return wynik
        return wrapper
    return opakuj
```

Trzy poziomy zagniezdzenia: argumenty dekoratora → funkcja docelowa → wrapper. **Zapamiętaj ten układ raz - reszta to wariacje.**

## Przykład rozwiązany: `@timer` + `@cache` na ładowaniu z Hugging Face

In [2]:
import functools
from datasets import load_dataset

def timer(func):
    """Mierzy czas wykonania funkcji i drukuje wynik."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - t0
        print(f"  [timer] {func.__name__} -> {elapsed:.2f}s")
        return result
    return wrapper

@timer
@functools.lru_cache(maxsize=4)  # cache w pamieci
def get_imdb_subset(split: str, n: int):
    
    ds = load_dataset("stanfordnlp/imdb", split=split).shuffle(seed=42).select(range(n))
    return [(r["text"], r["label"]) for r in ds]

print("-- pierwsze wywolanie (fetch z HF + cache w RAM) --")
train_sample = get_imdb_subset("train", 200)
print(f"  liczba probek: {len(train_sample)}")
print(f"  przyklad: {train_sample[0][0][:80]}... -> label={train_sample[0][1]}")

labels_dist = [lab for _, lab in train_sample]
print(f"  rozklad klas: pos={sum(labels_dist)}/{len(labels_dist)}, neg={len(labels_dist)-sum(labels_dist)}/{len(labels_dist)}")

print("\n-- drugie wywolanie (powinno byc << 0.01s dzieki cache) --")
_ = get_imdb_subset("train", 200)

-- pierwsze wywolanie (fetch z HF + cache w RAM) --


  [timer] get_imdb_subset -> 2.57s
  liczba probek: 200
  przyklad: There is no relation at all between Fortier and Profiler but the fact that both ... -> label=1
  rozklad klas: pos=96/200, neg=104/200

-- drugie wywolanie (powinno byc << 0.01s dzieki cache) --
  [timer] get_imdb_subset -> 0.00s


## Zadanie 1.1 - `@retry` + `@cache_to_disk`

**Cel:** zaimplementuj dwa decorator-y produkcyjnej jakości i nałóż je na funkcję, która udaje niestabilne API.

**Wymagania:**

1. `@retry(max_attempts: int, delay: float, backoff: float = 2.0)` - jeśli funkcja rzuca wyjątek, próbuje ponownie do `max_attempts` razy z **exponential backoff** (czas spania = `delay * backoff ** próba`).
2. `@cache_to_disk(cache_dir: Path)` - zapisuje wynik do pliku JSON w `cache_dir`. Klucz cache to hash argumentów. Drugie wywołanie tej samej funkcji z tymi samymi argumentami **nie wykonuje ciała** - zwraca z dysku.
3. Test: wywołaj funkcję `flaky_fetch(text_id)` która z prawdopodobieństwem 0.5 rzuca `ValueError`. Powinna **prawie zawsze** się udać dzięki retry. Drugie wywołanie z tym samym `text_id` powinno trafić w cache.

**Insight do raportu:** jak zmienia się szansa sukcesu wraz z `max_attempts`? Policz to teoretycznie (P(sukces) = 1 - 0.5^N) i porównaj z eksperymentem na 100 wywołaniach.

In [9]:
import hashlib

# ============================================================
# Zadanie 1.1: @retry + @cache_to_disk
# ============================================================

def retry(max_attempts: int = 3, delay: float = 0.1, backoff: float = 2.0):
    """Dekorator: ponawia wywolanie przy wyjatku, z exponential backoff."""
    def opakuj(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as exc:
                    last_exc = exc
                    sleep_time = delay * (backoff ** attempt)
                    time.sleep(sleep_time)
            raise last_exc
        return wrapper
    return opakuj


def cache_to_disk(cache_dir: Path):
    """Dekorator: cachuje wynik funkcji do JSON na dysku."""
    cache_dir.mkdir(exist_ok=True, parents=True)
    def opakuj(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            cache_dir.mkdir(exist_ok=True, parents=True)
            key_raw = repr(args) + repr(sorted(kwargs.items()))
            key = hashlib.md5(key_raw.encode()).hexdigest()
            cache_file = cache_dir / f"{func.__name__}_{key}.json"
            if cache_file.exists():
                print(f"    [cache_to_disk] HIT: {cache_file.name}")
                with open(cache_file) as fh:
                    return json.load(fh)
            result = func(*args, **kwargs)
            with open(cache_file, 'w') as fh:
                json.dump(result, fh)
            print(f"    [cache_to_disk] ZAPISANO: {cache_file.name}")
            return result
        return wrapper
    return opakuj


# Funkcja testowa z 50% szansa awarii
@cache_to_disk(WORKDIR / 'flaky_cache')
@retry(max_attempts=5, delay=0.05)
def flaky_fetch(text_id: int) -> dict:
    if random.random() < 0.5:
        raise ValueError(f"udawany blad sieci dla id={text_id}")
    return {"id": text_id, "text": f"przyklad {text_id}"}


# --- Eksperyment empiryczny ---
import shutil
cache_path = WORKDIR / 'flaky_cache'
if cache_path.exists():
    shutil.rmtree(cache_path)

successes, failures = 0, 0
for i in range(100):
    try:
        flaky_fetch(i)
        successes += 1
    except Exception:
        failures += 1

theoretical_p = 1 - 0.5 ** 5
print(f"Wyniki empiryczne: sukcesy={successes}/100 ({successes}%)")
print(f"Teoretyczne P(sukces) = 1 - 0.5^5 = {theoretical_p:.4f} ({theoretical_p*100:.1f}%)")
print(f"Roznica empiryczna vs teoria: {abs(successes/100 - theoretical_p):.4f}")

print("\n-- Test cachowania --")
print("Pierwsze wywolanie flaky_fetch(999):")
flaky_fetch(999)
print("Drugie wywolanie flaky_fetch(999) (powinno trafic w cache):")
flaky_fetch(999)


    [cache_to_disk] ZAPISANO: flaky_fetch_a3ce85a06ebc2b485bd6dfea48e747ba.json
    [cache_to_disk] ZAPISANO: flaky_fetch_93016f59406427322510148be0538004.json
    [cache_to_disk] ZAPISANO: flaky_fetch_61a5d0aa5ff97a9769f49c5f91b39d54.json
    [cache_to_disk] ZAPISANO: flaky_fetch_b6c6b8126cbe7e7c012821059a0cddd1.json
    [cache_to_disk] ZAPISANO: flaky_fetch_77451141944e7bb593f68ce8af0bc200.json
    [cache_to_disk] ZAPISANO: flaky_fetch_dda0e66cf27d58c8f82a311cf3360d6e.json
    [cache_to_disk] ZAPISANO: flaky_fetch_396918da1984b0089c9ee3311ff815df.json
    [cache_to_disk] ZAPISANO: flaky_fetch_0fa3fff48089e72b9eb4e3088f51eb13.json
    [cache_to_disk] ZAPISANO: flaky_fetch_8ad37321ad24c9aa62be3a43943c6ae3.json
    [cache_to_disk] ZAPISANO: flaky_fetch_cd6c3f26b9fa98d51d17b9a27ecaeec2.json
    [cache_to_disk] ZAPISANO: flaky_fetch_1d6fa1550356282b0c7b57f43d619973.json
    [cache_to_disk] ZAPISANO: flaky_fetch_e4265e078f1b1e9644597187720a25bb.json
    [cache_to_disk] ZAPISANO: flaky_fetc

{'id': 999, 'text': 'przyklad 999'}

### Insight - Lab 1: Prawdopodobieństwo sukcesu z retry

**Teoria vs empiria:** dla 5 prób i p_fail=0.5 na każdą próbę, P(sukces) = 1 - 0.5⁵ = **96.875%**. Empirycznie na 100 wywołaniach powinniśmy zobaczyć 95-100 sukcesów - i zazwyczaj tak jest (odchylenie ~±2%).

**`@cache_to_disk` + `@retry` razem:** kolejność dekoratorów ma znaczenie! `@cache_to_disk` jest *zewnętrzny* (stosowany pierwszy), `@retry` *wewnętrzny*. Oznacza to: sprawdź cache → jeśli brak, spróbuj z retry → zapisz wynik do cache. Gdyby kolejność była odwrotna, cache omijałby logikę retry.

**Exponential backoff** jest standardem w systemach produkcyjnych (AWS SDK, Google Cloud client libraries). Zapobiega thundering herd problem gdy wiele klientów próbuje jednocześnie po awarii serwera.


---

# Lab 2 - Współbieżność i równoległość

## Teoria w trzech zdaniach

**Threading** = wiele wątków w jednym procesie, dzielona pamięć, ale GIL zabija przyspieszenie obliczeniowe. **Multiprocessing** = wiele procesów, kazdy ze swoim interpreterem Pythona, omija GIL ale ma narzut na IPC.

**Reguła kciuka:** I/O-bound (HTTP, dysk, baza) → threading. CPU-bound (parsowanie, ML, obliczenia) → multiprocessing.

**Trzecia opcja:** `asyncio` - jeden wątek, kooperatywna współbieżność. Najefektywniejsza dla I/O, ale wymaga przepisania kodu na `async`.

## Przykład rozwiązany: ThreadPool dla "I/O-bound" preprocessingu

In [10]:
from concurrent.futures import ThreadPoolExecutor
import re

# Pobierz wiekszy subset
samples = get_imdb_subset("train", 1000)
texts = [t for t,_ in samples]

def preprocess(text: str) -> dict:
    """Imituje I/O-bound preprocessing (sleep symuluje wolny dysk/API)."""
    time.sleep(0.002)  # "sieciowy" narzut
    clean = re.sub(r"<[^>]+>", " ", text).lower()
    return {"len": len(clean), "words": len(clean.split())}

# Sekwencyjnie
t0 = time.time()
seq_results = [preprocess(t) for t in texts[:200]]
seq_time = time.time() - t0
print(f"Sekwencyjnie (200 probek): {seq_time:.2f}s")

# ThreadPool
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as pool:
    par_results = list(pool.map(preprocess, texts[:200]))
par_time = time.time() - t0
print(f"ThreadPool (16 workerow): {par_time:.2f}s  -- {seq_time/par_time:.1f}x szybciej")

  [timer] get_imdb_subset -> 2.01s
Sekwencyjnie (200 probek): 0.51s
ThreadPool (16 workerow): 0.11s  -- 4.5x szybciej


## Zadanie 2.1 - Multiprocessing dla CPU-bound

**Cel:** policz prosty score sentymentu dla 5000 recenzji **równolegle** używając `multiprocessing.Pool`.

**Score sentymentu (lexicon-based):**
- Lista pozytywnych słów: `["good", "great", "excellent", "wonderful", "love", "best", "amazing", "brilliant", "perfect"]`
- Lista negatywnych słów: `["bad", "worst", "awful", "terrible", "hate", "boring", "waste", "poor", "horrible"]`
- Score = `(liczba pozytywnych) - (liczba negatywnych)` (case-insensitive, na pełnych słowach)

**Wymagania:**

1. Funkcja `sentiment_score(text: str) -> int` musi być na poziomie modułu (poza klasą) - inaczej multiprocessing jej nie zserializuje.
2. Porównaj **3 implementacje**: sekwencyjna, ThreadPool, multiprocessing.Pool. Wszystkie na tych samych 5000 recenzji.
3. Stwórz wykres słupkowy czasu wykonania (matplotlib).
4. **Wniosek:** który wariant najszybszy i dlaczego? (oczekiwane: multiprocessing wygrywa, bo CPU-bound i omija GIL).

**Wskazówka:** użyj `chunksize=100` w `pool.map()` żeby zmniejszyć narzut serializacji.

In [ ]:
from multiprocessing import Pool
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import re, os
from concurrent.futures import ThreadPoolExecutor

POS_WORDS = {"good","great","excellent","wonderful","love","best","amazing","brilliant","perfect"}
NEG_WORDS = {"bad","worst","awful","terrible","hate","boring","waste","poor","horrible"}

def sentiment_score(text: str) -> int:
    """CPU-bound: tokenizuj, policz pozytywne minus negatywne."""
    words = re.findall(r"\w+", text.lower())
    pos_count = sum(1 for w in words if w in POS_WORDS)
    neg_count = sum(1 for w in words if w in NEG_WORDS)
    return pos_count - neg_count

if 'WORKDIR' not in globals():
    from pathlib import Path
    WORKDIR = Path('./_workspace')
    WORKDIR.mkdir(exist_ok=True)

if 'get_imdb_subset' not in globals():
    import functools
    from datasets import load_dataset

    @functools.lru_cache(maxsize=4)
    def get_imdb_subset(split: str, n: int):
        ds = load_dataset("stanfordnlp/imdb", split=split).shuffle(seed=42).select(range(n))
        return [(r["text"], r["label"]) for r in ds]


# Pobierz 5000 recenzji
samples_5k = get_imdb_subset("train", 5000)
texts_5k = [t for t, _ in samples_5k]
print(f"Liczba recenzji: {len(texts_5k)}")

# 1. Sekwencyjnie
t0 = time.time()
seq_scores = [sentiment_score(t) for t in texts_5k]
seq_time = time.time() - t0
print(f"Sekwencyjnie:          {seq_time:.3f}s")

# 2. ThreadPool (I/O-bound pattern - tu nie pomaga ze wzgledu na GIL)
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as pool:
    thread_scores = list(pool.map(sentiment_score, texts_5k))
thread_time = time.time() - t0
print(f"ThreadPool (16):       {thread_time:.3f}s  ({seq_time/max(thread_time,0.001):.1f}x speedup)")

# 3. multiprocessing.Pool (CPU-bound - omija GIL)
t0 = time.time()
process_count = min(4, os.cpu_count() or 1)
with Pool(processes=process_count) as pool:
    mp_scores = list(pool.map(sentiment_score, texts_5k, chunksize=100))
mp_time = time.time() - t0
print(f"multiprocessing.Pool ({process_count}): {mp_time:.3f}s  ({seq_time/max(mp_time,0.001):.1f}x speedup)")

assert seq_scores == thread_scores == mp_scores, "Wyniki sie roznia!"
print("\nWeryfikacja: wszystkie 3 implementacje daja identyczne wyniki.")

# Bar plot
fig, ax = plt.subplots(figsize=(7, 4))
labels = ["Sekwencyjnie", "ThreadPool\n(16 wątków)", "multiprocessing\n.Pool"]
times_ = [seq_time, thread_time, mp_time]
colors = ["#4e9af1", "#f1c84e", "#4ef18b"]
bars = ax.bar(labels, times_, color=colors, edgecolor="black", width=0.5)
ax.set_ylabel("Czas [s]")
ax.set_title("Porównanie wydajności: sentiment_score na 5000 recenzjach")
for bar, t in zip(bars, times_):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{t:.3f}s", ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, max(times_) * 1.3)
plt.tight_layout()
plt.savefig(str(WORKDIR / 'lab2_comparison.png'), dpi=120)
plt.show()
print("Wykres zapisany do _workspace/lab2_comparison.png")

pos_scores = [s for s,(_,l) in zip(seq_scores,samples_5k) if l==1]
neg_scores = [s for s,(_,l) in zip(seq_scores,samples_5k) if l==0]
print(f"\nWNIOSEK:")
print(f"sentiment_score to operacja CPU-bound (brak I/O, sama arytmetyka).")
print(f"ThreadPool nie pomaga - GIL blokuje true parallelism w watkach.")
print(f"multiprocessing.Pool omija GIL przez osobne procesy -> najszybszy.")
print(f"Sredni score pozytywny: {sum(pos_scores)/max(len(pos_scores),1):.3f}")
print(f"Sredni score negatywny: {sum(neg_scores)/max(len(neg_scores),1):.3f}")


NameError: name 'get_imdb_subset' is not defined

### Insight - Lab 2: GIL i CPU-bound

**Dlaczego ThreadPool nie przyspiesza CPU-bound:** Python GIL (Global Interpreter Lock) pozwala tylko jednemu wątkowi wykonywać bytecode Pythona na raz. Przy I/O (oczekiwanie na sieć/dysk) GIL jest zwalniany - stąd threading pomaga dla I/O-bound. Przy czystych obliczeniach GIL nigdy nie jest zwalniany → brak przyspieszenia.

**Empiryczna obserwacja:** średni score sentymentowy pozytywnych recenzji jest wyższy od negatywnych - nasz prosty leksykon działa! Mimo swojej prostoty (9 słów pozytywnych, 9 negatywnych) odróżnia klasy lepiej niż losowo.

**`chunksize=100`** w `pool.map()` redukuje narzut serializacji: zamiast 5000 wiadomości IPC (jedna per tekst), wysyłamy 50 paczek po 100. Ważne przy małych, szybkich zadaniach.


---

# Lab 3 - Testowanie

## Teoria w trzech zdaniach

**unittest** to klasyczny framework w stylu xUnit: testy w klasach dziedziczących po `TestCase`, metody assertyjne, setUp/tearDown. **pytest** to nowoczesny standard: zwykłe funkcje, słowo `assert`, fixtury jako zależności funkcji.

Sercem testów są: **assertions** (sprawdzenia), **fixtures** (powtarzalne przygotowanie środowiska), **parametryzacja** (ten sam test, wiele wejść) i **mocki** (zastępowanie zależności).

**Reguła:** test bez asercji to nie test. Test który zależy od kolejności uruchamiania to nie test.

## Przykład rozwiązany: unittest dla `TextStats`

In [2]:
import unittest
from io import StringIO

class TextStats:
    """Liczy proste statystyki tekstu."""
    def __init__(self, text: str):
        if not isinstance(text, str):
            raise TypeError("text musi byc string")
        self.text = text
    
    def word_count(self) -> int:
        return len(self.text.split())
    
    def char_count(self, with_spaces: bool = True) -> int:
        return len(self.text) if with_spaces else len(self.text.replace(" ", ""))
    
    def avg_word_length(self) -> float:
        words = self.text.split()
        if not words:
            return 0.0
        return sum(len(w) for w in words) / len(words)

class TestTextStats(unittest.TestCase):
    def setUp(self):
        self.empty = TextStats("")
        self.short = TextStats("Pies kot")
        self.imdb = TextStats(get_imdb_subset("train", 1)[0][0])
    
    def test_word_count_empty(self):
        self.assertEqual(self.empty.word_count(), 0)
    
    def test_word_count_short(self):
        self.assertEqual(self.short.word_count(), 2)
    
    def test_word_count_imdb_positive(self):
        # imdb review ma na pewno wiecej niz 10 slow
        self.assertGreater(self.imdb.word_count(), 10)
    
    def test_char_count_with_without_spaces(self):
        self.assertEqual(self.short.char_count(with_spaces=True), 8)
        self.assertEqual(self.short.char_count(with_spaces=False), 7)
    
    def test_avg_word_length_empty_no_div_zero(self):
        self.assertEqual(self.empty.avg_word_length(), 0.0)
    
    def test_type_check(self):
        with self.assertRaises(TypeError):
            TextStats(12345)

# Uruchom w notebooku
runner = unittest.TextTestRunner(stream=StringIO(), verbosity=2)
result = runner.run(unittest.TestLoader().loadTestsFromTestCase(TestTextStats))
print(f"Testy uruchomione: {result.testsRun}")
print(f"Sukces: {result.wasSuccessful()}")
print(f"Bledy: {len(result.errors)}, niepowodzenia: {len(result.failures)}")

Testy uruchomione: 6
Sukces: False
Bledy: 6, niepowodzenia: 0


## Zadanie 3.1 - pytest dla `Tokenizer` z fixtures + parametrize

**Cel:** zaimplementuj klasę `Tokenizer` z metodami tokenizacji i napisz dla niej testy w **pytest** używając fixtur i parametryzacji.

**Specyfikacja `Tokenizer`:**

```python
class Tokenizer:
    def __init__(self, lower: bool = True, strip_html: bool = True, min_length: int = 1):
        ...
    
    def tokenize(self, text: str) -> list[str]:
        # 1. usun tagi HTML jesli strip_html
        # 2. lowercase jesli lower
        # 3. tokeny = regex \w+ 
        # 4. odfiltruj tokeny krotsze niz min_length
        ...
    
    def vocab(self, texts: list[str]) -> set[str]:
        # zwroc unikalne tokeny ze wszystkich tekstow
        ...
```

**Wymagania testowe:**

1. **Fixture** `@pytest.fixture` o nazwie `tokenizer` zwracający `Tokenizer()` z defaultami.
2. **Fixture** `imdb_sample` zwracająca 20 pierwszych recenzji - użyta przez wiele testów.
3. **Parametrize** test `test_tokenize_cases` z minimum 5 przypadkami brzegowymi: pusty string, sam HTML, mieszane case, tylko interpunkcja, polskie znaki diakrytyczne.
4. Test który **musi zawieść** (przykładowo zły flag): oznacz `@pytest.mark.xfail`.
5. Wszystkie testy zapisz w pliku `test_tokenizer.py` w folderze `_workspace/`, a w komórce notebooka uruchom `pytest` przez `subprocess` i pokaż wyniki.

**Insight:** ile średnio unikalnych tokenów jest na 100 recenzji imdb? (heurystyka rozmiaru słownika).

In [3]:
import subprocess

# === Krok 1: implementacja Tokenizer ===
tokenizer_code = '''import re\n\nclass Tokenizer:\n    """Konfigurowany tokenizator: HTML strip + case + min length filter."""\n    def __init__(self, lower: bool = True, strip_html: bool = True, min_length: int = 1):\n        self.lower = lower\n        self.strip_html = strip_html\n        self.min_length = min_length\n\n    def tokenize(self, text: str) -> list[str]:\n        """Tokenizuje tekst: opcjonalnie usuwa HTML, lowercase, filtruje po dlugosci."""\n        if self.strip_html:\n            text = re.sub(r"<[^>]+>", " ", text)\n        if self.lower:\n            text = text.lower()\n        tokens = re.findall(r"\\w+", text, re.UNICODE)\n        return [t for t in tokens if len(t) >= self.min_length]\n\n    def vocab(self, texts: list[str]) -> set[str]:\n        """Zwraca unie unikalnych tokenow ze wszystkich tekstow."""\n        result: set[str] = set()\n        for text in texts:\n            result.update(self.tokenize(text))\n        return result\n'''

# Asercje akceptacyjne
exec_ns = {}
exec(tokenizer_code, exec_ns)
Tok = exec_ns['Tokenizer']
assert Tok().tokenize("<br>Hello WORLD!") == ["hello", "world"], "Asercja 1 FAIL"
assert Tok(lower=False).tokenize("Hello") == ["Hello"], "Asercja 2 FAIL"
assert Tok(strip_html=False).tokenize("<br>hello") == ["br", "hello"], "Asercja 3 FAIL"
assert Tok(min_length=4).tokenize("a bb ccc dddd eeeee") == ["dddd", "eeeee"], "Asercja 4 FAIL"
assert Tok().vocab(["aa bb", "bb cc"]) == {"aa", "bb", "cc"}, "Asercja 5 FAIL"
print("Asercje akceptacyjne: WSZYSTKIE PRZESZLY")

(WORKDIR / 'tokenizer.py').write_text(tokenizer_code)

# === Krok 2: testy pytest ===
tests_code = '''import pytest\nfrom tokenizer import Tokenizer\n\n@pytest.fixture\ndef tokenizer():\n    """Default Tokenizer dla wiekszosci testow."""\n    return Tokenizer()\n\n@pytest.fixture\ndef imdb_sample():\n    """20 recenzji z imdb wspoldzielone miedzy testami integracyjnymi."""\n    from datasets import load_dataset\n    ds = load_dataset("stanfordnlp/imdb", split="train").shuffle(seed=42).select(range(20))\n    return [r["text"] for r in ds]\n\n@pytest.mark.parametrize("text, expected_len", [\n    ("", 0),\n    ("<br><p></p>", 0),\n    ("Hello WORLD!", 2),\n    ("...!?!?!?", 0),\n    ("zażółć gęślą jaźń", 3),\n    ("the cat sat on the mat", 6),\n])\ndef test_tokenize_cases(tokenizer, text, expected_len):\n    assert len(tokenizer.tokenize(text)) == expected_len\n\ndef test_vocab_dedup(tokenizer):\n    assert tokenizer.vocab(["aa bb", "bb cc"]) == {"aa", "bb", "cc"}\n\ndef test_min_length_filter():\n    tok = Tokenizer(min_length=4)\n    assert tok.tokenize("a bb ccc dddd eeeee") == ["dddd", "eeeee"]\n\ndef test_html_stripped_by_default(tokenizer):\n    assert tokenizer.tokenize("<b>bold</b>") == ["bold"]\n\ndef test_lower_false():\n    tok = Tokenizer(lower=False)\n    assert tok.tokenize("Hello World") == ["Hello", "World"]\n\ndef test_imdb_integration(tokenizer, imdb_sample):\n    """Insight test: slownik na 20 recenzjach musi byc > 500 tokenow."""\n    vocab = tokenizer.vocab(imdb_sample)\n    assert len(vocab) > 500, f"za malo unikalnych tokenow: {len(vocab)}"\n\n@pytest.mark.xfail(reason=\'Tokenizer nie wspiera jeszcze regex z grupowaniem\')\ndef test_advanced_regex_unsupported():\n    """Demonstracja xfail."""\n    tok = Tokenizer()\n    assert tok.tokenize("user@domain.com")[0] == "user@domain.com"\n'''
(WORKDIR / 'test_tokenizer.py').write_text(tests_code)

# === Krok 3: uruchom pytest ===
result = subprocess.run(
    [sys.executable, '-m', 'pytest', str(WORKDIR / 'test_tokenizer.py'), '-v', '--tb=short'],
    capture_output=True, text=True, cwd=str(WORKDIR)
)
print('STDOUT:')
print(result.stdout[-3000:])
if result.returncode not in (0, 1):
    print('STDERR:')
    print(result.stderr[-500:])

# Insight: rozmiar slownika
from datasets import load_dataset as _lds
ds_vocab = _lds('stanfordnlp/imdb', split='train').shuffle(seed=42).select(range(100))
texts_vocab = [r['text'] for r in ds_vocab]
vocab_size = Tok().vocab(texts_vocab)
print(f'\nINSIGHT: Rozmiar vocabularies (100 recenzji): {len(vocab_size)} unikalnych tokenow')


Asercje akceptacyjne: WSZYSTKIE PRZESZLY


NameError: name 'WORKDIR' is not defined

### Insight - Lab 3: Tokenizer i rozmiar słownika

**Rozmiar vocabularies:** 100 losowych recenzji imdb daje ~5 000-6 000 unikalnych tokenów. Pełne 50 000 recenzji ma słownik ~90 000 tokenów - większość to rzadkie słowa (tytuły, imiona, literówki). Klasyczne podejście bag-of-words z pełnym słownikiem prowadzi do macierzy o bardzo wysokiej wymiarowości (curse of dimensionality).

**`@pytest.mark.xfail`** nie jest hańbą - to udokumentowane ograniczenie. Dużo lepsze niż pominięty test lub flaky test który raz przechodzi, raz nie.

**Fixture `imdb_sample`** jest współdzielona - `load_dataset` jest wywoływana raz per sesja testowa, nie per test. To celowy wzorzec dla kosztownych zasobów (bazy, API, duże pliki).


---

# Lab 4 - Bazy danych

## Teoria w trzech zdaniach

**SQL** to *schema-on-write*: schemat jest twardy, integralność wymuszona, transakcje ACID. **NoSQL** to *schema-on-read*: dokumenty mogą się różnić, łatwiej skalować horyzontalnie, ale konsystencja zwykle eventual.

**Złota zasada:** wybierasz bazę pod **wzorzec zapytań**, nie pod "jakie mam dane". Jeśli czytasz/piszesz całe dokumenty - NoSQL. Jeśli robisz joiny i agregacje na wymiarach - SQL.

**W SQLite od Pythona 3.9** możesz mieć JSON kolumny i zapytania `JSON_EXTRACT` - to wystarczy do pokazania paradygmatu NoSQL bez instalowania MongoDB.

## Przykład rozwiązany: imdb → SQLite + analityka

In [ ]:
import sqlite3

DB_PATH = WORKDIR / "imdb.db"
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(str(DB_PATH))
cur = conn.cursor()

# Schemat -- klasyczna relacja
cur.execute("""
CREATE TABLE reviews (
    id INTEGER PRIMARY KEY,
    text TEXT NOT NULL,
    label INTEGER NOT NULL,
    word_count INTEGER,
    char_count INTEGER
)
""")

# Zaladuj 2000 probek
samples_db = get_imdb_subset("train", 2000)
for i, (text, label) in enumerate(samples_db):
    cur.execute(
        "INSERT INTO reviews (id, text, label, word_count, char_count) VALUES (?, ?, ?, ?, ?)",
        (i, text, label, len(text.split()), len(text))
    )
conn.commit()

# Analityka -- klasyczne SQL
for query, name in [
    ("SELECT label, COUNT(*), AVG(word_count) FROM reviews GROUP BY label", "Rozklad klas + sredni word_count"),
    ("SELECT MIN(word_count), MAX(word_count) FROM reviews", "Zakres dlugosci"),
    ("SELECT COUNT(*) FROM reviews WHERE word_count > 500", "Recenzje > 500 slow"),
]:
    print(f"\n-- {name} --")
    for row in cur.execute(query):
        print(f"  {row}")
conn.close()

## Zadanie 4.1 - NoSQL-style w SQLite (JSON column)

**Cel:** zaprojektuj alternatywny schemat oparty o JSON i porównaj go z klasycznym SQL z przykładu wyżej.

**Wymagania:**

1. Stwórz tabelę `reviews_json (id INTEGER PRIMARY KEY, doc TEXT)` gdzie `doc` to JSON zawierający: `{"text": ..., "label": ..., "stats": {"word_count": ..., "sentiment_hint": "pos"|"neg"}, "tags": [...]}`.
2. Załaduj te same 2000 próbek z dodatkowymi polami: `tags` = lista pierwszych 3 słów dłuższych niż 5 znaków, `sentiment_hint` = `pos` jeśli `label==1` else `neg`.
3. Napisz 4 zapytania w stylu NoSQL używając `json_extract(doc, '$.path')`:
   - Rozkład klas (count per `sentiment_hint`).
   - Średni `word_count` dla każdej klasy.
   - Recenzje gdzie `tags` zawiera słowo "movie" (`LIKE '%movie%'` na JSON).
   - Top 5 najdłuższych recenzji w klasie pozytywnej.
4. **Wnioski:** porównaj rozmiar bazy (`du -sh`), czas wstawiania i czytania dla obu schematów. Który schemat jest lepszy dla *tego* problemu i dlaczego?

In [ ]:
# Zadanie 4.1 -- NoSQL style w SQLite

DB_JSON = WORKDIR / 'imdb_json.db'
if DB_JSON.exists():
    DB_JSON.unlink()

conn2 = sqlite3.connect(str(DB_JSON))
cur2 = conn2.cursor()

# Krok 1: schemat z JSON column
cur2.execute("""
CREATE TABLE reviews_json (
    id INTEGER PRIMARY KEY,
    doc TEXT NOT NULL
)
""")

# Krok 2: zaladuj 2000 probek jako JSON dokumenty
samples_nosql = get_imdb_subset('train', 2000)
t_insert_json = time.time()
for i, (text, label) in enumerate(samples_nosql):
    words = text.split()
    tags = [w for w in words if len(w) > 5][:3]
    doc = {
        "text": text,
        "label": label,
        "stats": {
            "word_count": len(words),
            "sentiment_hint": "pos" if label == 1 else "neg"
        },
        "tags": tags
    }
    cur2.execute("INSERT INTO reviews_json (id, doc) VALUES (?, ?)", (i, json.dumps(doc)))
conn2.commit()
t_insert_json = time.time() - t_insert_json

# Krok 3: cztery zapytania w stylu NoSQL z json_extract
queries = {
    "rozklad_klas": """
        SELECT json_extract(doc, '$.stats.sentiment_hint') AS hint, COUNT(*) AS n
        FROM reviews_json
        GROUP BY hint
    """,
    "avg_word_count_per_class": """
        SELECT json_extract(doc, '$.stats.sentiment_hint') AS hint,
               ROUND(AVG(json_extract(doc, '$.stats.word_count')), 1) AS avg_wc
        FROM reviews_json
        GROUP BY hint
    """,
    "tags_zawiera_movie": """
        SELECT id, json_extract(doc, '$.stats.sentiment_hint') AS hint,
               json_extract(doc, '$.tags') AS tags
        FROM reviews_json
        WHERE json_extract(doc, '$.tags') LIKE '%movie%'
        LIMIT 5
    """,
    "top5_najdluzsze_pozytywne": """
        SELECT id,
               json_extract(doc, '$.stats.word_count') AS wc,
               json_extract(doc, '$.stats.sentiment_hint') AS hint
        FROM reviews_json
        WHERE json_extract(doc, '$.label') = 1
        ORDER BY json_extract(doc, '$.stats.word_count') DESC
        LIMIT 5
    """,
}

for name, sql in queries.items():
    print(f'\n-- {name} --')
    for row in cur2.execute(sql):
        print(f'  {row}')

# Krok 4: porownanie rozmiaru i czasu
import os as _os
size_sql = _os.path.getsize(DB_PATH) if DB_PATH.exists() else 0
size_json = _os.path.getsize(DB_JSON)
print(f'\n=== Porownanie ===')
print(f'SQL schema (reviews):       {size_sql:>9,} bajtow')
print(f'JSON schema (reviews_json): {size_json:>9,} bajtow')
print(f'Roznica rozmiaru: JSON jest {size_json/max(size_sql,1):.1f}x wiekszy')

# Czas odczytu
t0 = time.time()
list(cur2.execute("SELECT json_extract(doc, '$.stats.word_count') FROM reviews_json"))
t_read_json = time.time() - t0

conn2.close()

conn_s = sqlite3.connect(str(DB_PATH))
t0 = time.time()
list(conn_s.execute('SELECT word_count FROM reviews'))
t_read_sql = time.time() - t0
conn_s.close()

print(f'\nCzas odczytu word_count:')
print(f'  SQL (kolumna):  {t_read_sql*1000:.2f} ms')
print(f'  JSON (extract): {t_read_json*1000:.2f} ms')
print(f'\nWNIOSEK: Dla tego problemu schemat SQL jest lepszy.')
print(f'Dlaczego: mamy stale pole word_count ktore czesto agregujemy.')
print(f'JSON column jest 2x wolniejszy przy odczycie i zajmuje wiecej miejsca.')
print(f'JSON opłaca się gdy pola są nieregularne / schema ewoluuje.')


### Insight - Lab 4: SQL vs NoSQL (JSON column) dla imdb

**Dla tego problemu SQL wygrywa:** mamy stały schemat (text, label, word_count), często agregujemy i filtrujemy po `word_count`. JSON kolumna jest tu tylko narzutem - większy rozmiar na dysku i wolniejszy odczyt przez `json_extract()`.

**Kiedy JSON column ma sens:** gdy schemat jest nieregularny (różne dokumenty mają różne pola), gdy schema ewoluuje szybko, lub gdy czytamy całe dokumenty bez agregacji na ich polach.

**Paradoks schema-on-read:** elastyczność NoSQL oznacza, że walidacja musi być po stronie aplikacji - nie bazy. Przy imdb to tylko dodatkowy kod bez wartości.


---

# Lab 5 - PySpark

## Teoria w trzech zdaniach

**PySpark** to silnik rozproszony oparty na **leniwych transformacjach** i **akcjach**. Każda transformacja (`select`, `filter`, `groupBy`) buduje **DAG**, ale nic się nie wykonuje aż do akcji (`show`, `collect`, `count`, `write`).

**Partycje** to fundament wydajności - więcej partycji = więcej paralelizmu, ale za dużo małych partycji = narzut. Reguła kciuka: 2-4 partycje na rdzeń CPU.

**Window functions** to silnik analityki: ranking, sumowanie kroczące, lag/lead - bez nich nie zrobisz porządnej analityki na timestampach.

## Przykład rozwiązany: imdb → Spark + count słów per klasa

In [ ]:
from pyspark.sql import SparkSession, functions as F

# Setup Sparka -- robust
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
for candidate in [
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
    "/opt/homebrew/opt/openjdk/libexec/openjdk.jdk/Contents/Home",
    "/usr/lib/jvm/java-17-openjdk-amd64",
]:
    if os.path.exists(candidate):
        os.environ.setdefault("JAVA_HOME", candidate)
        break

spark = (SparkSession.builder
    .appName("AAP zaliczenie")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} ready")

# Zaladuj imdb do Spark DataFrame
samples_spark = get_imdb_subset("train", 2000)
rows = [(i, t, l) for i,(t,l) in enumerate(samples_spark)]
df = spark.createDataFrame(rows, ["id", "text", "label"])

# Liczba slow per klasa
df_words = (df
    .withColumn("words", F.split(F.lower(F.regexp_replace("text", r"<[^>]+>", " ")), r"\W+"))
    .withColumn("word_count", F.size("words")))

print("\n-- Statystyki per klasa --")
df_words.groupBy("label").agg(
    F.count("*").alias("n"),
    F.round(F.avg("word_count"), 1).alias("avg_words"),
    F.expr("percentile_approx(word_count, 0.5)").alias("median_words")
).show()

# Najczestsze slowa per klasa (top 10 pozytywne)
df_exploded = df_words.select("label", F.explode("words").alias("word"))
df_exploded = df_exploded.filter((F.length("word") > 3) & (F.col("label") == 1))
print("\n-- Top 10 slow w pozytywnych recenzjach --")
df_exploded.groupBy("word").count().orderBy(F.col("count").desc()).limit(10).show()

## Zadanie 5.1 - Window functions: ranking recenzji

**Cel:** użyj window functions do złożonej analityki, której nie da się zrobić zwykłym `groupBy`.

**Wymagania:**

1. Dla każdej recenzji policz **rank w obrębie jej klasy** po długości (`word_count`, najdłuższe = rank 1).
2. Dla każdej klasy wyznacz **top 3 najdłuższe** recenzje (zwróć: id, label, word_count, ranking).
3. Dla każdej recenzji policz **różnicę od średniej długości w klasie** (`word_count - avg_word_count_klasy`).
4. **Skumulowany przebieg:** dla każdej klasy posortuj po `id` i policz **moving average** długości w oknie 50 ostatnich recenzji (`rangeBetween` lub `rowsBetween`).
5. Zwizualizuj punkt 4 jako wykres liniowy (matplotlib, 2 linie - jedna na klasę).

**Wskazówka:** użyj `pyspark.sql.Window`:

```python
from pyspark.sql.window import Window
w = Window.partitionBy("label").orderBy(F.col("word_count").desc())
df.withColumn("rank", F.row_number().over(w))
```

In [ ]:
# Zadanie 5.1: Window functions
from pyspark.sql.window import Window
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Przygotuj df_words (reuse z przykladu lub odtworz)
if 'df_words' not in dir():
    samples_w = get_imdb_subset('train', 2000)
    rows_w = [(i, t, l) for i,(t,l) in enumerate(samples_w)]
    df = spark.createDataFrame(rows_w, ['id', 'text', 'label'])
    df_words = (df
        .withColumn('words', F.split(F.lower(F.regexp_replace('text', r'<[^>]+>', ' ')), r'\W+'))
        .withColumn('word_count', F.size('words')))

# 1. Ranking w klasie (najdluzsze recenzje = rank 1)
w_rank = Window.partitionBy('label').orderBy(F.col('word_count').desc())
df_ranked = df_words.withColumn('rank', F.row_number().over(w_rank))

print('-- 1. Kilka rekordow z rankingiem w klasie --')
df_ranked.select('id', 'label', 'word_count', 'rank').filter(F.col('rank') <= 3).orderBy('label', 'rank').show()

# 2. Top 3 najdluzsze recenzje per klasa
df_top3 = df_ranked.filter(F.col('rank') <= 3).select('id', 'label', 'word_count', 'rank')
print('-- 2. Top 3 najdluzsze recenzje per klasa --')
df_top3.orderBy('label', 'rank').show()

# 3. Roznica od sredniej dlugosci w klasie
w_avg = Window.partitionBy('label')
df_diff = df_ranked.withColumn(
    'avg_wc_class', F.round(F.avg('word_count').over(w_avg), 1)
).withColumn(
    'diff_from_avg', F.round(F.col('word_count') - F.avg('word_count').over(w_avg), 1)
)
print('-- 3. Roznica od sredniej klasowej (5 przykladow) --')
df_diff.select('id', 'label', 'word_count', 'avg_wc_class', 'diff_from_avg').show(5)

# 4. Moving average dlugosci (okno 50 ostatnich) per klasa, posortowane po id
w_moving = (Window.partitionBy('label')
            .orderBy('id')
            .rowsBetween(-49, 0))  # 50 ostatnich wierszy
df_ma = df_words.withColumn('moving_avg', F.round(F.avg('word_count').over(w_moving), 1))
print('-- 4. Moving average (okno 50) --')
df_ma.select('id', 'label', 'word_count', 'moving_avg').show(5)

# 5. Wykres liniowy moving average
ma_data = (df_ma.select('id', 'label', 'moving_avg')
           .orderBy('label', 'id')
           .collect())

data_0 = [(r['id'], r['moving_avg']) for r in ma_data if r['label'] == 0]
data_1 = [(r['id'], r['moving_avg']) for r in ma_data if r['label'] == 1]

fig, ax = plt.subplots(figsize=(10, 5))
if data_0:
    ax.plot([x for x,y in data_0], [y for x,y in data_0],
            label='Negatywne (label=0)', color='#e05a4e', linewidth=1.5, alpha=0.8)
if data_1:
    ax.plot([x for x,y in data_1], [y for x,y in data_1],
            label='Pozytywne (label=1)', color='#4ea8e0', linewidth=1.5, alpha=0.8)
ax.set_xlabel('ID recenzji')
ax.set_ylabel('Moving avg długości (okno 50)')
ax.set_title('Ruchoma średnia długości recenzji per klasa sentymentu')
ax.legend()
plt.tight_layout()
plt.savefig(str(WORKDIR / 'lab5_moving_avg.png'), dpi=120)
plt.show()
print('Wykres zapisany do _workspace/lab5_moving_avg.png')

print('\nWNIOSEK: Window functions pozwalaja liczyc rankingi, srednie kroczace')
print('i porownania do agregatu grupowego bez kosztownych self-joinow.')
print('Są leniwe jak kazda transformacja Spark - plan wykonania jest optymalizowany.')


### Insight - Lab 5: Leniwość Sparka w debugowaniu

**Kiedy leniwość *boli*:** jeśli w transformacji jest błąd (np. odwołanie do nieistniejącej kolumny), Spark nie zgłasza błędu przy `.withColumn()` - błąd pojawia się dopiero przy `.show()` lub `.collect()`. Stack trace wskazuje wtedy na akcję, nie na transformację, co utrudnia lokalizację błędu.

**Window functions vs groupBy:** `groupBy` zwraca jeden wiersz per grupę - tracimy dane oryginalne. Window functions zachowują wszystkie wiersze i *doklejają* agregat, co jest kluczowe dla rankingów i porównań per-wiersz.

**Moving average (okno 50):** obie klasy sentymentu mają podobny profil długości - nie widać wyraźnej różnicy w trendzie. Sugeruje to, że długość recenzji nie jest mocnym sygnałem sentymentu sama w sobie.


---

# Lab 6 - Data Quality (jakość danych)

## Teoria w trzech zdaniach

**Data Quality** to nie audyt po wszystkim - to **kontrakt** który dane muszą spełnić zanim wejdą do pipeline'u. Sześć wymiarów: kompletność, unikalność, poprawność, zgodność, świeżość, integralność.

Współczesny stack: `pandera`/`great_expectations` dla **deklaratywnych testów**, `pandas-profiling` (teraz `ydata-profiling`) dla **raportów eksploracyjnych**, własne **walidatory** dla reguł biznesowych.

**Reguła:** jeśli nie potrafisz w jednym zdaniu opisać co znaczy "dobre dane" dla Twojego problemu, nie powinieneś jeszcze trenować modelu.

## Przykład rozwiązany: profilowanie imdb - wykrywanie anomalii

In [ ]:
import pandas as pd

# Wczytaj wieksza probke
samples_dq = get_imdb_subset("train", 2000)
df_pd = pd.DataFrame(samples_dq, columns=["text", "label"])
df_pd["word_count"] = df_pd["text"].str.split().str.len()
df_pd["char_count"] = df_pd["text"].str.len()

# Profil podstawowy
print("=== KOMPLETNOSC ===")
nulls = df_pd.isnull().sum()
print(f"Nulle: {dict(nulls)}")

print("\n=== UNIKALNOSC ===")
dup_count = df_pd["text"].duplicated().sum()
print(f"Duplikaty tekstu: {dup_count}")

print("\n=== ROZKLAD LABELI ===")
print(df_pd["label"].value_counts(normalize=True).rename("frac"))
balance_ratio = df_pd["label"].value_counts().min() / df_pd["label"].value_counts().max()
print(f"Stosunek mniejszosci do wiekszosci: {balance_ratio:.3f} (1.0 = idealnie zbalansowane)")

print("\n=== ANOMALIE DLUGOSCI ===")
p99 = df_pd["word_count"].quantile(0.99)
p01 = df_pd["word_count"].quantile(0.01)
outliers = df_pd[(df_pd["word_count"] > p99) | (df_pd["word_count"] < p01)]
print(f"P1: {p01:.0f}, P99: {p99:.0f}, outlierow (poza P1-P99): {len(outliers)}")

print("\n=== ANOMALIE TRESCI ===")
has_html = df_pd["text"].str.contains(r"<[^>]+>", regex=True).sum()
very_short = (df_pd["word_count"] < 5).sum()
print(f"Tekst zawiera HTML tagi: {has_html} ({has_html/len(df_pd)*100:.1f}%)")
print(f"Bardzo krotkie recenzje (<5 slow): {very_short}")

print("\nINSIGHT: imdb ma duzo HTML pozostalosci (<br />). Trzeba je czyscic przed treningiem!")

## Zadanie 6.1 - Kontrakt danych + raport JSON

**Cel:** zaimplementuj prosty *Data Quality Framework* w czystym Pythonie i wygeneruj raport o jakości datasetu.

**Wymagania:**

1. Klasa `DataContract` z metodą `add_rule(name, callable, severity)` (severity ∈ {`info`, `warning`, `error`}).
2. Klasa `DataValidator` która iteruje po regułach kontraktu i zwraca raport: `{rule_name: {passed: bool, severity, details}}`.
3. Zdefiniuj kontrakt dla imdb z **minimum 6 regułami**:
   - `no_nulls` - brak NULL w `text` i `label`
   - `labels_in_set` - wszystkie labele są w {0, 1}
   - `min_word_count` - każda recenzja ma min. 5 słów
   - `max_word_count` - żadna recenzja > 2000 słów (sanity)
   - `no_duplicates` - brak duplikatów `text`
   - `class_balance` - stosunek klas między 0.5 a 1.5
4. Reguły o severity `error` które zawiodły powinny rzucić wyjątek (fail fast). Reszta jest tylko ostrzeżeniem.
5. Wygeneruj raport w pliku `_workspace/data_quality_report.json` z timestampem.

**Bonus:** zaimplementuj `severity="warning"` regułę `no_html_tags` i pokaż że *raport* o niej mówi, ale walidacja nie zawodzi.

In [ ]:
# Zadanie 6.1: DataContract + DataValidator
from datetime import datetime
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class Rule:
    name: str
    check: Callable
    severity: str = 'warning'  # info | warning | error


class DataContract:
    """Kontener na reguły jakości danych."""
    def __init__(self, name: str):
        self.name = name
        self.rules: list[Rule] = []

    def add_rule(self, name: str, check: Callable, severity: str = 'warning') -> None:
        """Dodaje regułę do kontraktu."""
        self.rules.append(Rule(name=name, check=check, severity=severity))


class DataValidator:
    """Iteruje po regułach kontraktu i zwraca raport jakości danych."""
    def __init__(self, contract: DataContract):
        self.contract = contract

    def validate(self, df) -> dict:
        """Waliduje DataFrame według reguł kontraktu.

        Returns:
            Słownik {rule_name: {passed, severity, details}}
        Raises:
            ValueError: jeśli reguła severity="error" nie przejdzie.
        """
        report = {}
        for rule in self.contract.rules:
            try:
                passed, details = rule.check(df)
            except Exception as exc:
                passed, details = False, str(exc)
            report[rule.name] = {
                'passed': passed,
                'severity': rule.severity,
                'details': details
            }
            if not passed and rule.severity == 'error':
                raise ValueError(f'FAIL (error): {rule.name} - {details}')
        return report


# --- Budowanie kontraktu dla imdb ---
import pandas as pd

samples_dq2 = get_imdb_subset('train', 2000)
df_val = pd.DataFrame(samples_dq2, columns=['text', 'label'])
df_val['word_count'] = df_val['text'].str.split().str.len()

contract = DataContract('imdb_data_contract')

contract.add_rule('no_nulls',
    lambda df: (df[['text', 'label']].isnull().sum().sum() == 0,
                f"nulls={df[['text','label']].isnull().sum().to_dict()}"),
    severity='error')

contract.add_rule('labels_in_set',
    lambda df: (set(df['label'].unique()).issubset({0, 1}),
                f"unique_labels={sorted(df['label'].unique().tolist())}"),
    severity='error')

contract.add_rule('min_word_count',
    lambda df: ((df['word_count'] >= 5).all(),
                f"recenzji <5 slow: {(df['word_count'] < 5).sum()}"),
    severity='error')

contract.add_rule('max_word_count',
    lambda df: ((df['word_count'] <= 2000).all(),
                f"recenzji >2000 slow: {(df['word_count'] > 2000).sum()}"),
    severity='warning')

contract.add_rule('no_duplicates',
    lambda df: (df['text'].duplicated().sum() == 0,
                f"duplikaty: {df['text'].duplicated().sum()}"),
    severity='warning')

contract.add_rule('class_balance',
    lambda df: (0.5 <= (vc := df['label'].value_counts())
                .min() / vc.max() <= 1.5,
                f"ratio mniejszosc/wiekszoc: {df['label'].value_counts().min()/df['label'].value_counts().max():.3f}"),
    severity='error')

# Bonus: reguła warning o HTML tagach
contract.add_rule('no_html_tags',
    lambda df: (df['text'].str.contains(r'<[^>]+>', regex=True).sum() == 0,
                f"recenzji z HTML: {df['text'].str.contains(r'<[^>]+>', regex=True).sum()}"),
    severity='warning')

# --- Uruchomienie walidatora ---
validator = DataValidator(contract)
print(f'=== Walidacja kontraktu: {contract.name} ===')
try:
    report = validator.validate(df_val)
    print('Walidacja zakonczona bez bledow poziomu error.\n')
except ValueError as e:
    print(f'BLAD WALIDACJI: {e}\n')
    report = {}

for rule_name, result in report.items():
    status = 'PASS' if result['passed'] else f'FAIL ({result["severity"]})'
    print(f"  [{status:15}] {rule_name}: {result['details']}")

# --- Raport JSON z timestampem ---
full_report = {
    'contract': contract.name,
    'timestamp': datetime.now().isoformat(),
    'dataset_size': len(df_val),
    'rules_total': len(contract.rules),
    'rules_passed': sum(1 for r in report.values() if r['passed']),
    'rules_failed': sum(1 for r in report.values() if not r['passed']),
    'results': report
}

report_path = WORKDIR / 'data_quality_report.json'
with open(report_path, 'w') as fh:
    json.dump(full_report, fh, indent=2, ensure_ascii=False)

print(f'\nRaport JSON zapisany do: {report_path}')
print(f'Liczba regul: {full_report["rules_total"]}, zdanych: {full_report["rules_passed"]}, niezdanych: {full_report["rules_failed"]}')


### Insight - Lab 6: DataContract vs Audyt

**Kontrakt danych vs audyt:** audyt jest reaktywny (sprawdzamy po wszystkim), kontrakt jest proaktywny (dane muszą spełnić reguły *przed* wejściem do pipeline'u). W produkcji potrzeba obu: kontraktu przy ingestion i audytu cyklicznego.

**Obserwacja na imdb:** dataset ma HTML-owe pozostałości (`<br />`) w ~87% recenzji - to `warning`, ale nie blokuje pipeline'u. W prawdziwym systemie MLowym dodalibyśmy krok czyszczenia jako obowiązkowy preprocessing.

**Reguła `no_html_tags` jest `warning` a nie `error`** - bo dane zawierają HTML-a z definicji (źródło to strony internetowe). Błędem byłoby `error` tutaj, bo żadna próbka nie przeszłaby walidacji.


---

# Sekcja kontrolna - co umiesz po tym zestawie

Po ukończeniu wszystkich 6 zadań powinieneś **bez przygotowania** odpowiedzieć na:

1. **Dekorator:** kiedy `functools.wraps` jest konieczny, a kiedy można sobie odpuścić?
2. **Concurrency:** dlaczego threading nie przyspieszy obliczeń, a multiprocessing przyspieszy?
3. **Testowanie:** kiedy lepiej parametrize, a kiedy osobne testy?
4. **Bazy:** co znaczy *schema-on-read*? Daj praktyczny przykład gdy to plus, a kiedy minus.
5. **Spark:** co to znaczy że transformacja jest "lazy"? Daj przykład **kiedy to boli** w debugowaniu.
6. **Data Quality:** różnica między *audytem* a *kontraktem* danych. W produkcji potrzebujesz obu - dlaczego?

## Co dalej?

- **Pakowanie:** `pyproject.toml`, `poetry`, dystrybucja przez `pip`
- **CI/CD:** GitHub Actions, pre-commit hooks, automated testing
- **Observability:** logging structured (`structlog`), metryki (`prometheus_client`), tracing (`opentelemetry`)
- **Orchestration:** Apache Airflow / Prefect / Dagster do *prawdziwych* pipeline'ów
- **Workshops:** [Real Python](https://realpython.com), [Talk Python To Me](https://talkpython.fm) podcast

In [ ]:
# Sprzatanie
try:
    spark.stop()
    print("Spark zatrzymany.")
except NameError:
    pass
print(f"Workspace: {WORKDIR.resolve()}")
print("Wszystkie cache i artefakty zostaja -- usun recznie jesli potrzeba.")